In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

# ============================================================
# NB_Silver_To_Gold
# Microsoft Fabric - Insurance Gold Layer
#
# Source      : LH_Silver
# Destination : LH_Gold
#
# Creates:
#   dim_customer
#   dim_policy
#   fact_claim
#   fact_payment
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    row_number,
    year,
    month,
    when
)
from pyspark.sql.window import Window


# ============================================================
# 1. CONFIGURATION
# ============================================================

SILVER_LAKEHOUSE = "LH_Silver"
GOLD_LAKEHOUSE   = "LH_Gold"
SCHEMA           = "dbo"


print("================================================")
print("SILVER -> GOLD PROCESS")
print("================================================")


# ============================================================
# 2. READ SILVER TABLES
# ============================================================

customers = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.silver_customers"
)

policies = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.silver_policies"
)

claims = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.silver_claims"
)

payments = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.silver_payments"
)


# ============================================================
# 3. READ REJECT TABLES
# ============================================================

reject_customers = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.reject_customers"
)

reject_policies = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.reject_policies"
)

reject_claims = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.reject_claims"
)

reject_payments = spark.table(
    f"{SILVER_LAKEHOUSE}.{SCHEMA}.reject_payments"
)


print("Silver and reject tables loaded successfully.")


# ============================================================
# 4. REMOVE REJECTED RECORDS
# ============================================================

valid_customers = (
    customers
    .join(
        reject_customers.select("customer_id").distinct(),
        "customer_id",
        "left_anti"
    )
)

valid_policies = (
    policies
    .join(
        reject_policies.select("policy_id").distinct(),
        "policy_id",
        "left_anti"
    )
)

valid_claims = (
    claims
    .join(
        reject_claims.select("claim_id").distinct(),
        "claim_id",
        "left_anti"
    )
)

valid_payments = (
    payments
    .join(
        reject_payments.select("payment_id").distinct(),
        "payment_id",
        "left_anti"
    )
)


print("\nVALID SILVER RECORDS")
print("----------------------------------------------")
print(f"Customers : {valid_customers.count()}")
print(f"Policies  : {valid_policies.count()}")
print(f"Claims    : {valid_claims.count()}")
print(f"Payments  : {valid_payments.count()}")


# ============================================================
# 5. DIM_CUSTOMER
# ============================================================
#
# Create deterministic surrogate customer key.
#
# Since this lab rebuilds Gold using CREATE/REPLACE,
# ordering by customer_id gives a stable numeric key.
# ============================================================

customer_window = Window.orderBy("customer_id")

dim_customer = (
    valid_customers

    .withColumn(
        "customer_key",
        row_number().over(customer_window)
    )

    .select(
        "customer_key",
        "customer_id",
        "first_name",
        "last_name",
        "date_of_birth",
        "email",
        "phone",
        "address",
        "city",
        "state",
        "zip_code",
        "customer_status",
        "created_date"
    )

    .withColumn(
        "_gold_processed_ts",
        current_timestamp()
    )
)


# ============================================================
# 6. DIM_POLICY
# ============================================================

policy_window = Window.orderBy("policy_id")

dim_policy_base = (
    valid_policies

    .withColumn(
        "policy_key",
        row_number().over(policy_window)
    )
)


# Bring customer surrogate key into policy dimension

customer_key_lookup = (
    dim_customer
    .select(
        "customer_id",
        "customer_key"
    )
)


dim_policy = (
    dim_policy_base

    .join(
        customer_key_lookup,
        "customer_id",
        "inner"
    )

    .select(
        "policy_key",
        "policy_id",

        "customer_key",
        "customer_id",

        "product_type",
        "policy_start_date",
        "policy_end_date",
        "annual_premium",
        "coverage_limit",
        "deductible",
        "policy_status",
        "agent_id",
        "last_updated"
    )

    .withColumn(
        "_gold_processed_ts",
        current_timestamp()
    )
)


# ============================================================
# 7. PREPARE DIMENSION LOOKUPS
# ============================================================

policy_key_lookup = (
    dim_policy
    .select(
        "policy_id",
        "policy_key"
    )
)

customer_key_lookup = (
    dim_customer
    .select(
        "customer_id",
        "customer_key"
    )
)


# ============================================================
# 8. FACT_CLAIM
# ============================================================

claim_window = Window.orderBy("claim_id")

fact_claim = (
    valid_claims

    .withColumn(
        "claim_key",
        row_number().over(claim_window)
    )

    .join(
        policy_key_lookup,
        "policy_id",
        "inner"
    )

    .join(
        customer_key_lookup,
        "customer_id",
        "inner"
    )

    # Useful analytical attributes
    .withColumn(
        "claim_year",
        year(col("claim_date"))
    )

    .withColumn(
        "claim_month",
        month(col("claim_date"))
    )

    # Useful business metric
    .withColumn(
        "unapproved_amount",
        col("claim_amount") -
        F.coalesce(
            col("approved_amount"),
            lit(0.0)
        )
    )

    .withColumn(
        "approval_percentage",
        when(
            col("claim_amount") > 0,
            (
                F.coalesce(
                    col("approved_amount"),
                    lit(0.0)
                )
                /
                col("claim_amount")
            ) * 100
        )
        .otherwise(0.0)
    )

    .select(
        "claim_key",
        "claim_id",

        "policy_key",
        "policy_id",

        "customer_key",
        "customer_id",

        "claim_date",
        "incident_date",
        "claim_year",
        "claim_month",

        "claim_type",
        "claim_status",

        "claim_amount",
        "approved_amount",
        "unapproved_amount",
        "approval_percentage",

        "description",
        "reported_channel",
        "adjuster_id"
    )

    .withColumn(
        "_gold_processed_ts",
        current_timestamp()
    )
)


# ============================================================
# 9. FACT_PAYMENT
# ============================================================

payment_window = Window.orderBy("payment_id")

# claim lookup gives us claim surrogate key
claim_key_lookup = (
    fact_claim
    .select(
        "claim_id",
        "claim_key"
    )
)


fact_payment = (
    valid_payments

    .withColumn(
        "payment_key",
        row_number().over(payment_window)
    )

    .join(
        policy_key_lookup,
        "policy_id",
        "inner"
    )

    .join(
        customer_key_lookup,
        "customer_id",
        "inner"
    )

    # Some payments may not have a claim_id,
    # therefore use LEFT JOIN here.
    .join(
        claim_key_lookup,
        "claim_id",
        "left"
    )

    .withColumn(
        "payment_year",
        year(col("payment_date"))
    )

    .withColumn(
        "payment_month",
        month(col("payment_date"))
    )

    .select(
        "payment_key",
        "payment_id",

        "policy_key",
        "policy_id",

        "customer_key",
        "customer_id",

        "claim_key",
        "claim_id",

        "payment_date",
        "payment_year",
        "payment_month",

        "payment_type",
        "payment_amount",
        "payment_method",
        "payment_status",
        "transaction_reference"
    )

    .withColumn(
        "_gold_processed_ts",
        current_timestamp()
    )
)


# ============================================================
# 10. CREATE / REPLACE GOLD TABLE FUNCTION
# ============================================================

def create_or_replace_gold_table(
    dataframe,
    table_name
):

    full_name = (
        f"{GOLD_LAKEHOUSE}."
        f"{SCHEMA}."
        f"{table_name}"
    )

    print(
        f"Creating/Replacing: {full_name}"
    )

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(full_name)
    )

    print(
        f"SUCCESS: {full_name}"
    )


# ============================================================
# 11. WRITE GOLD TABLES
# ============================================================

print("\n================================================")
print("WRITING GOLD TABLES")
print("================================================")


create_or_replace_gold_table(
    dim_customer,
    "dim_customer"
)

create_or_replace_gold_table(
    dim_policy,
    "dim_policy"
)

create_or_replace_gold_table(
    fact_claim,
    "fact_claim"
)

create_or_replace_gold_table(
    fact_payment,
    "fact_payment"
)


# ============================================================
# 12. READ GOLD TABLES BACK
# ============================================================

gold_customers = spark.table(
    "LH_Gold.dbo.dim_customer"
)

gold_policies = spark.table(
    "LH_Gold.dbo.dim_policy"
)

gold_claims = spark.table(
    "LH_Gold.dbo.fact_claim"
)

gold_payments = spark.table(
    "LH_Gold.dbo.fact_payment"
)


# ============================================================
# 13. GOLD VALIDATION
# ============================================================

print("\n================================================")
print("GOLD VALIDATION")
print("================================================")

print(
    f"dim_customer : {gold_customers.count()}"
)

print(
    f"dim_policy   : {gold_policies.count()}"
)

print(
    f"fact_claim   : {gold_claims.count()}"
)

print(
    f"fact_payment : {gold_payments.count()}"
)


# ============================================================
# 14. BUSINESS METRIC VALIDATION
# ============================================================

print("\n================================================")
print("BUSINESS METRICS")
print("================================================")


claims_metrics = (
    gold_claims
    .agg(

        F.count("*")
        .alias("claim_count"),

        F.sum("claim_amount")
        .alias("total_claim_amount"),

        F.sum("approved_amount")
        .alias("total_approved_amount"),

        F.avg("claim_amount")
        .alias("average_claim_amount")
    )
)

display(claims_metrics)


payment_metrics = (
    gold_payments
    .agg(

        F.count("*")
        .alias("payment_count"),

        F.sum("payment_amount")
        .alias("total_payment_amount"),

        F.avg("payment_amount")
        .alias("average_payment_amount")
    )
)

display(payment_metrics)


# ============================================================
# 15. SAMPLE FACT DATA
# ============================================================

print("\nSample Gold Claims:")

display(
    gold_claims
    .orderBy("claim_key")
    .limit(10)
)


# ============================================================
# 16. COMPLETION
# ============================================================

print("\n================================================")
print("SILVER -> GOLD PROCESS COMPLETED")
print("================================================")

print("Created/Replaced:")

print(
    "LH_Gold.dbo.dim_customer"
)

print(
    "LH_Gold.dbo.dim_policy"
)

print(
    "LH_Gold.dbo.fact_claim"
)

print(
    "LH_Gold.dbo.fact_payment"
)

print("================================================")

StatementMeta(, f5f810b8-e79a-452a-b61f-934563436286, 3, Finished, Available, Finished, False)

SILVER -> GOLD PROCESS
Silver and reject tables loaded successfully.

VALID SILVER RECORDS
----------------------------------------------
Customers : 499
Policies  : 749
Claims    : 1198
Payments  : 1499

WRITING GOLD TABLES
Creating/Replacing: LH_Gold.dbo.dim_customer
SUCCESS: LH_Gold.dbo.dim_customer
Creating/Replacing: LH_Gold.dbo.dim_policy
SUCCESS: LH_Gold.dbo.dim_policy
Creating/Replacing: LH_Gold.dbo.fact_claim
SUCCESS: LH_Gold.dbo.fact_claim
Creating/Replacing: LH_Gold.dbo.fact_payment
SUCCESS: LH_Gold.dbo.fact_payment

GOLD VALIDATION
dim_customer : 499
dim_policy   : 748
fact_claim   : 1197
fact_payment : 1497

BUSINESS METRICS


SynapseWidget(Synapse.DataFrame, b5c7182c-cd3b-4fa1-be62-c83b7655957b)

SynapseWidget(Synapse.DataFrame, 2eb7592a-6fce-44ec-8081-9bb01c60c6e4)


Sample Gold Claims:


SynapseWidget(Synapse.DataFrame, 6e8779be-0b9e-4441-9a31-1ef54a9d0a23)


SILVER -> GOLD PROCESS COMPLETED
Created/Replaced:
LH_Gold.dbo.dim_customer
LH_Gold.dbo.dim_policy
LH_Gold.dbo.fact_claim
LH_Gold.dbo.fact_payment
